In [8]:
#this is a details tutotial and implementation of the GO embedding only model
#trainning and testing on mESC TFs+500 STRING dataset
import os
import random
import numpy as np
import pandas as pd
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import GATConv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# 1. Configuration
# ============================================================

SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OBO_PATH = "go.obo"
TERM_PATH = "terms.tsv"

BP_PATH = (
    "mESC_BP_GO_mapped.csv"
)

PAIR_PATH = (
    "/home/featurize/work/cs-main/DeepTGI/dataset/scRNA-Seq/STRING Dataset/mESC/TFs+500/AllPair.csv"
)

HIDDEN_DIM = 128
GO_EMBED_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 2
DROPOUT = 0.2

BATCH_SIZE = 32
EPOCHS = 80
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
N_SPLITS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. Parse GO ontology
#    Original graph direction: child -> parent
# ============================================================

def parse_go_obo(file_path):
    go_graph = nx.DiGraph()
    go_name_dict = {}
    go_def_dict = {}

    current_id = None
    current_name = ""
    current_def = ""
    parents = []
    is_obsolete = False

    def save_term():
        if current_id and not is_obsolete:
            go_graph.add_node(current_id)

            if current_name:
                go_name_dict[current_id] = current_name

            if current_def:
                go_def_dict[current_id] = current_def

            for parent in parents:
                go_graph.add_edge(current_id, parent)

    with open(file_path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()

            if line == "[Term]":
                save_term()

                current_id = None
                current_name = ""
                current_def = ""
                parents = []
                is_obsolete = False

            elif line.startswith("["):
                save_term()

                current_id = None
                current_name = ""
                current_def = ""
                parents = []
                is_obsolete = False

            elif line.startswith("id: GO:"):
                current_id = line.split("id: ", 1)[1]

            elif line.startswith("name:") and current_id:
                current_name = line.split("name:", 1)[1].strip()

            elif line.startswith("def:") and current_id:
                current_def = line.split("def:", 1)[1].strip()

            elif line.startswith("is_obsolete: true"):
                is_obsolete = True

            elif line.startswith("is_a: GO:") and current_id:
                parent_id = line.split("is_a:", 1)[1].split()[0]
                parents.append(parent_id)

        save_term()

    # Remove edges whose parent terms were not retained
    valid_nodes = set(go_graph.nodes())
    go_graph.remove_edges_from([
        (child, parent)
        for child, parent in go_graph.edges()
        if parent not in valid_nodes
    ])

    if not nx.is_directed_acyclic_graph(go_graph):
        raise ValueError("GO ontology graph is not a DAG.")

    return go_graph, go_name_dict, go_def_dict


# ============================================================
# 3. Build GO hierarchy features
#    One multi-hot vector per GO node:
#    the node itself + all ancestors
# ============================================================

def get_go_ancestor_vector(go_graph, all_go_terms):
    go2idx = {
        go: i for i, go in enumerate(all_go_terms)
    }

    n_go = len(all_go_terms)

    features = torch.zeros(
        (n_go, n_go), dtype=torch.float32
    )

    for go in all_go_terms:
        i = go2idx[go]
        features[i, i] = 1.0

        ancestors = nx.ancestors(go_graph, go)

        for ancestor in ancestors:
            if ancestor in go2idx:
                features[i, go2idx[ancestor]] = 1.0

    return features


# ============================================================
# 4. GOTreeEncoder
#    GO hierarchy features -> GAT -> GO node embeddings
# ============================================================

class GOTreeEncoder(nn.Module):

    def __init__(
        self,
        obo_path=OBO_PATH,
        hidden_dim=HIDDEN_DIM,
        out_dim=GO_EMBED_DIM
    ):
        super().__init__()

        (
            self.go_graph,
            self.go_name_dict,
            self.go_def_dict
        ) = parse_go_obo(obo_path)

        # One consistent node ordering for features, edges and lookup
        self.all_terms = list(self.go_graph.nodes())

        self.go2idx = {
            go: i for i, go in enumerate(self.all_terms)
        }

        self.idx2go = {
            i: go for go, i in self.go2idx.items()
        }

        hierarchy_features = get_go_ancestor_vector(
            self.go_graph, self.all_terms
        )

        self.register_buffer(
            "hierarchy_features", hierarchy_features
        )

        # Reverse child -> parent into parent -> child.
        # This propagates information from ancestors to descendants.
        edges = [
            (
                self.go2idx[parent],
                self.go2idx[child]
            )
            for child, parent in self.go_graph.edges()
        ]

        if len(edges) > 0:
            edge_index = torch.tensor(
                edges, dtype=torch.long
            ).t().contiguous()
        else:
            edge_index = torch.empty(
                (2, 0), dtype=torch.long
            )

        self.register_buffer("edge_index", edge_index)

        self.struct_proj = nn.Linear(
            len(self.all_terms), hidden_dim
        )

        self.gat1 = GATConv(
            hidden_dim,
            hidden_dim,
            heads=4,
            concat=False,
            add_self_loops=True
        )

        self.gat2 = GATConv(
            hidden_dim,
            out_dim,
            heads=4,
            concat=False,
            add_self_loops=True
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self):
        x = self.struct_proj(self.hierarchy_features)

        x = self.gat1(x, self.edge_index)
        x = self.norm1(F.gelu(x))
        x = self.dropout(x)

        x = self.gat2(x, self.edge_index)
        x = self.norm2(x)

        # [number of GO nodes, GO embedding dimension]
        return x

    def get_go_embeddings(self):
        return self.forward()


# ============================================================
# 5. Map genes to GO node indices
#
# Assumed BP matrix:
#   rows    = GO IDs
#   columns = gene symbols
#   values  = 0/1 or nonnegative annotation values
# ============================================================

def build_gene_go_mapping(bp_path, go2idx):

    bp = pd.read_csv(bp_path, index_col=0)

    # Normalize row identifiers
    bp.index = bp.index.astype(str).str.strip()
    bp.columns = bp.columns.astype(str).str.strip()

    gene2go_idx = {}

    for gene in bp.columns:

        values = pd.to_numeric(
            bp[gene], errors="coerce"
        ).fillna(0).to_numpy()

        go_indices = []

        for go_id, value in zip(bp.index, values):

            if value > 0 and go_id in go2idx:
                go_indices.append(go2idx[go_id])

        gene2go_idx[gene] = sorted(set(go_indices))

    return gene2go_idx


# ============================================================
# 6. Dataset
# ============================================================

class TGIGODataset(Dataset):

    def __init__(self, pairs, labels, gene2go_idx):
        self.pairs = list(pairs)
        self.labels = np.asarray(labels)
        self.gene2go_idx = gene2go_idx

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        tf, gene = self.pairs[idx]

        tf_indices = self.gene2go_idx.get(tf, [])
        gene_indices = self.gene2go_idx.get(gene, [])

        return (
            tf_indices,
            gene_indices,
            float(self.labels[idx])
        )


# ============================================================
# 7. Collate variable-length GO lists
#    -1 denotes padding
# ============================================================

def go_collate_fn(batch):

    tf_lists, gene_lists, labels = zip(*batch)

    def pad_indices(index_lists):

        max_len = max(
            1, max(len(indices) for indices in index_lists)
        )

        padded = torch.full(
            (len(index_lists), max_len),
            -1,
            dtype=torch.long
        )

        for i, indices in enumerate(index_lists):

            if len(indices) > 0:
                padded[i, :len(indices)] = torch.tensor(
                    indices, dtype=torch.long
                )

        return padded

    return (
        pad_indices(tf_lists),
        pad_indices(gene_lists),
        torch.tensor(labels, dtype=torch.float32)
    )


# ============================================================
# 8. Transformer-based TF-gene interaction predictor
#
# TF GO embeddings   -> mean pooling -> token 1
# Gene GO embeddings -> mean pooling -> token 2
# Two tokens         -> Transformer
# Transformer output -> concatenation -> classifier
# ============================================================

class GOTransformer(nn.Module):

    def __init__(
        self,
        go_encoder,
        go_dim=GO_EMBED_DIM,
        n_heads=NUM_HEADS,
        n_layers=NUM_LAYERS,
        dropout=DROPOUT
    ):
        super().__init__()

        self.go_encoder = go_encoder

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=go_dim,
            nhead=n_heads,
            dim_feedforward=go_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.classifier = nn.Sequential(
            nn.Linear(go_dim * 2, go_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(go_dim, 1)
        )

    @staticmethod
    def mean_pool(go_embeddings, indices):

        # indices: [batch_size, max_num_go_terms]
        valid_mask = indices >= 0

        # Avoid invalid indexing for padding
        safe_indices = indices.clamp(min=0)

        selected = go_embeddings[safe_indices]

        mask = valid_mask.unsqueeze(-1).to(
            selected.dtype
        )

        summed = (selected * mask).sum(dim=1)

        counts = mask.sum(dim=1).clamp(min=1.0)

        pooled = summed / counts

        # Genes with no GO annotations receive zero vectors
        has_go = valid_mask.any(
            dim=1, keepdim=True
        )

        pooled = pooled * has_go.to(pooled.dtype)

        return pooled

    def forward(self, tf_go_idx, gene_go_idx):

        # Compute GO embeddings once per forward pass
        go_embeddings = self.go_encoder()

        tf_embedding = self.mean_pool(
            go_embeddings, tf_go_idx
        )

        gene_embedding = self.mean_pool(
            go_embeddings, gene_go_idx
        )

        # [B, 2, D]
        pair_tokens = torch.stack(
            [tf_embedding, gene_embedding],
            dim=1
        )

        encoded = self.transformer(pair_tokens)

        # [B, 2 * D]
        pair_embedding = encoded.reshape(
            encoded.size(0), -1
        )

        # [B], logits for BCEWithLogitsLoss
        logits = self.classifier(
            pair_embedding
        ).squeeze(-1)

        return logits


# ============================================================
# 9. Train and evaluate one fold
# ============================================================

def train_one_fold(
    model,
    train_loader,
    test_loader,
    device,
    epochs=EPOCHS
):

    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):

        model.train()
        total_loss = 0.0
        total_samples = 0

        for tf_idx, gene_idx, labels in train_loader:

            tf_idx = tf_idx.to(device)
            gene_idx = gene_idx.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(tf_idx, gene_idx)

            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

        if (epoch + 1) % 10 == 0 or epoch == 0:

            avg_loss = total_loss / max(
                total_samples, 1
            )

            print(
                f"Epoch {epoch + 1:03d}/{epochs}, "
                f"Loss={avg_loss:.4f}"
            )

    # Evaluation
    model.eval()

    all_labels = []
    all_scores = []

    with torch.no_grad():

        for tf_idx, gene_idx, labels in test_loader:

            logits = model(
                tf_idx.to(device),
                gene_idx.to(device)
            )

            probabilities = torch.sigmoid(logits)

            all_labels.extend(labels.numpy().tolist())
            all_scores.extend(
                probabilities.cpu().numpy().tolist()
            )

    all_labels = np.asarray(all_labels, dtype=int)
    all_scores = np.asarray(all_scores)

    auc = roc_auc_score(all_labels, all_scores)
    aupr = average_precision_score(
        all_labels, all_scores
    )

    # A fixed 0.5 threshold for classification metrics
    predictions = (all_scores >= 0.5).astype(int)

    acc = accuracy_score(all_labels, predictions)
    precision = precision_score(
        all_labels, predictions, zero_division=0
    )
    recall = recall_score(
        all_labels, predictions, zero_division=0
    )
    f1 = f1_score(
        all_labels, predictions, zero_division=0
    )

    return {
        "AUROC": auc,
        "AUPR": aupr,
        "ACC": acc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }


# ============================================================
# 10. Load data and five-fold cross-validation
# ============================================================

def main():

    print("Device:", DEVICE)

    print("Loading GO hierarchy...")

    # Parse once to build the gene annotation mapping
    go_encoder = GOTreeEncoder(
        obo_path=OBO_PATH,
        hidden_dim=HIDDEN_DIM,
        out_dim=GO_EMBED_DIM
    )

    print("Number of GO nodes:", len(go_encoder.all_terms))
    print("Number of GO edges:", go_encoder.edge_index.size(1))

    gene2go_idx = build_gene_go_mapping(
        BP_PATH,
        go_encoder.go2idx
    )

    annotated_genes = sum(
        len(indices) > 0
        for indices in gene2go_idx.values()
    )

    print("Genes in GO matrix:", len(gene2go_idx))
    print("Genes with GO annotations:", annotated_genes)

    print("Loading interaction pairs...")

    pairs_df = pd.read_csv(PAIR_PATH)

    # Support either already-named columns or three unnamed columns
    if not {"tf", "gene", "label"}.issubset(
        set(pairs_df.columns)
    ):
        if pairs_df.shape[1] != 3:
            raise ValueError(
                "AllPair.csv must contain exactly three columns: "
                "tf, gene, label."
            )

        pairs_df.columns = ["tf", "gene", "label"]

    pairs_df["tf"] = pairs_df["tf"].astype(str).str.strip()
    pairs_df["gene"] = pairs_df["gene"].astype(str).str.strip()

    pairs_df["label"] = pd.to_numeric(
        pairs_df["label"], errors="raise"
    ).astype(int)

    if not set(pairs_df["label"].unique()).issubset({0, 1}):
        raise ValueError("Interaction labels must be binary 0/1.")

    pairs = list(zip(pairs_df["tf"], pairs_df["gene"]))
    labels = pairs_df["label"].to_numpy()

    # Check coverage before training
    missing_tf = sum(
        tf not in gene2go_idx for tf, gene in pairs
    )
    missing_gene = sum(
        gene not in gene2go_idx for tf, gene in pairs
    )

    print("Total interaction pairs:", len(pairs))
    print("Positive pairs:", int(labels.sum()))
    print("Pairs with TF absent from GO matrix:", missing_tf)
    print("Pairs with gene absent from GO matrix:", missing_gene)

    if len(np.unique(labels)) != 2:
        raise ValueError(
            "Both positive and negative pairs are required."
        )

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    )

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(
        skf.split(np.zeros(len(labels)), labels),
        start=1
    ):

        print(f"\n========== Fold {fold}/{N_SPLITS} ==========")

        train_dataset = TGIGODataset(
            [pairs[i] for i in train_idx],
            labels[train_idx],
            gene2go_idx
        )

        test_dataset = TGIGODataset(
            [pairs[i] for i in test_idx],
            labels[test_idx],
            gene2go_idx
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=go_collate_fn,
            num_workers=0
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=go_collate_fn,
            num_workers=0
        )

        # Fresh GO encoder and predictor for each fold
        fold_encoder = GOTreeEncoder(
            obo_path=OBO_PATH,
            hidden_dim=HIDDEN_DIM,
            out_dim=GO_EMBED_DIM
        )

        model = GOTransformer(
            go_encoder=fold_encoder,
            go_dim=GO_EMBED_DIM,
            n_heads=NUM_HEADS,
            n_layers=NUM_LAYERS,
            dropout=DROPOUT
        )

        result = train_one_fold(
            model,
            train_loader,
            test_loader,
            DEVICE,
            epochs=EPOCHS
        )

        result["Fold"] = fold
        fold_results.append(result)

        print("Fold results:", result)

    # Summary
    results_df = pd.DataFrame(fold_results)

    metric_columns = [
        "AUROC", "AUPR", "ACC",
        "Precision", "Recall", "F1"
    ]

    print("\n========== Cross-validation results ==========")

    for metric in metric_columns:
        values = results_df[metric]

        print(
            f"{metric}: {values.mean():.4f} "
            f"+/- {values.std(ddof=1):.4f}"
        )

    results_df.to_csv(
        "GO_hierarchy_transformer_results.csv",
        index=False
    )

    print(
        "\nSaved results to "
        "GO_hierarchy_transformer_results.csv"
    )


if __name__ == "__main__":
    main()


Device: cuda
Loading GO hierarchy...
Number of GO nodes: 38245
Number of GO edges: 57824
Genes in GO matrix: 1120
Genes with GO annotations: 926
Loading interaction pairs...
Total interaction pairs: 15524
Positive pairs: 7762
Pairs with TF absent from GO matrix: 0
Pairs with gene absent from GO matrix: 0

========== Fold 1/5 ==========


/environment/miniconda3/lib/python3.11/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Epoch 001/80, Loss=0.6021
Epoch 010/80, Loss=0.3417
Epoch 020/80, Loss=0.2726
Epoch 030/80, Loss=0.2283
Epoch 040/80, Loss=0.1917
Epoch 050/80, Loss=0.1701
Epoch 060/80, Loss=0.1415
Epoch 070/80, Loss=0.1349
Epoch 080/80, Loss=0.1175
Fold results: {'AUROC': 0.9389110534316687, 'AUPR': 0.9219728910984244, 'ACC': 0.8853462157809984, 'Precision': 0.8394781622234827, 'Recall': 0.952994204764971, 'F1': 0.8926417370325693, 'Fold': 1}

========== Fold 2/5 ==========


/environment/miniconda3/lib/python3.11/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Epoch 001/80, Loss=0.6066
Epoch 010/80, Loss=0.3419
Epoch 020/80, Loss=0.2595
Epoch 030/80, Loss=0.2138
Epoch 040/80, Loss=0.1805
Epoch 050/80, Loss=0.1566
Epoch 060/80, Loss=0.1343
Epoch 070/80, Loss=0.1183
Epoch 080/80, Loss=0.1075
Fold results: {'AUROC': 0.9389728725911273, 'AUPR': 0.9267019025873235, 'ACC': 0.88341384863124, 'Precision': 0.8400913763563678, 'Recall': 0.9471989697359948, 'F1': 0.8904358353510896, 'Fold': 2}

========== Fold 3/5 ==========


...

Epoch 001/80, Loss=0.6171
Epoch 010/80, Loss=0.3346
Epoch 020/80, Loss=0.2671
Epoch 030/80, Loss=0.2234
Epoch 040/80, Loss=0.1869
Epoch 050/80, Loss=0.1573
Epoch 060/80, Loss=0.1350
Epoch 070/80, Loss=0.1193
Epoch 080/80, Loss=0.1065
Fold results: {'AUROC': 0.9367710317908139, 'AUPR': 0.9227114178976862, 'ACC': 0.8750402576489533, 'Precision': 0.8310580204778157, 'Recall': 0.9413659793814433, 'F1': 0.8827794561933535, 'Fold': 3}

========== Fold 4/5 ==========


/environment/miniconda3/lib/python3.11/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Epoch 001/80, Loss=0.6026
Epoch 010/80, Loss=0.3413
Epoch 020/80, Loss=0.2664
Epoch 030/80, Loss=0.2203
Epoch 040/80, Loss=0.1849
Epoch 050/80, Loss=0.1646
Epoch 060/80, Loss=0.1390
Epoch 070/80, Loss=0.1261
Epoch 080/80, Loss=0.1154
Fold results: {'AUROC': 0.9416937453946801, 'AUPR': 0.9248912746512629, 'ACC': 0.8830917874396135, 'Precision': 0.8368271954674221, 'Recall': 0.9516752577319587, 'F1': 0.890563762435936, 'Fold': 4}

========== Fold 5/5 ==========


/environment/miniconda3/lib/python3.11/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Epoch 001/80, Loss=0.6031
Epoch 010/80, Loss=0.3516
Epoch 020/80, Loss=0.2759
Epoch 030/80, Loss=0.2264
Epoch 040/80, Loss=0.1911
Epoch 050/80, Loss=0.1630
Epoch 060/80, Loss=0.1368
Epoch 070/80, Loss=0.1288
Epoch 080/80, Loss=0.1208
Fold results: {'AUROC': 0.9499185038925496, 'AUPR': 0.9363359916892593, 'ACC': 0.8959407216494846, 'Precision': 0.854178674351585, 'Recall': 0.9548969072164949, 'F1': 0.9017341040462428, 'Fold': 5}

========== Cross-validation results ==========
AUROC: 0.9413 +/- 0.0051
AUPR: 0.9265 +/- 0.0058
ACC: 0.8846 +/- 0.0075
Precision: 0.8403 +/- 0.0085
Recall: 0.9496 +/- 0.0054
F1: 0.8916 +/- 0.0068

Saved results to GO_hierarchy_transformer_results.csv


In [4]:
pip install torch_geometric scikit-learn

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 100.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 58.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 26.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 483.2/483.2 kB 35.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#generating terms.tsv file 
def extract_go_terms_from_obo(obo_path, output_path, namespace="biological_process"):
    """
    从 go.obo 中提取指定 namespace 的 GO term，
    输出为 load_go_id_list_from_compact_file() 可读取的格式。
    """
    go_ids = []
    current_id = None
    current_namespace = None
    is_obsolete = False

    def save_term():
        if (
            current_id
            and current_namespace == namespace
            and not is_obsolete
        ):
            go_ids.append(current_id)

    with open(obo_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line == "[Term]":
                # 保存上一个 GO term
                save_term()
                current_id = None
                current_namespace = None
                is_obsolete = False

            elif line.startswith("["):
                # 进入 [Typedef] 等其他区块
                save_term()
                current_id = None
                current_namespace = None
                is_obsolete = False

            elif line.startswith("id:"):
                current_id = line.split("id:", 1)[1].strip()

            elif line.startswith("namespace:"):
                current_namespace = line.split("namespace:", 1)[1].strip()

            elif line == "is_obsolete: true":
                is_obsolete = True

        # 保存文件末尾的最后一个 term
        save_term()

    # 去重并保持原始顺序
    go_ids = list(dict.fromkeys(go_ids))

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("##go-terms bp\n")
        for go_id in go_ids:
            f.write(go_id + "\n")

    print(f"提取完成：{len(go_ids)} 个 GO terms")
    print(f"输出文件：{output_path}")


if __name__ == "__main__":
    obo_path = "go.obo"       # 修改为你的 OBO 文件路径
    output_path = "terms.tsv"

    extract_go_terms_from_obo(
        obo_path=obo_path,
        output_path=output_path,
        namespace="biological_process"
    )

In [7]:
#map msigdb GO name to GO id
#gene_set	                                  go_id
#GOBP_2_OXOGLUTARATE_METABOLIC_PROCESS	   -> GO:0006103
#GOBP_3_UTR_MEDIATED_MRNA_DESTABILIZATION  -> GO:0061158
import csv
import re
import time
import requests

GMT_PATH = "m5.go.bp.v2026.1.Mm.symbols.gmt"
OBO_PATH = "go.obo"

GMT_PATH = "m5.go.bp.v2026.1.Mm.symbols.gmt"  
GMT_PATH = "/home/featurize/work/cs-main/DeepTGI/dataset/scRNA-Seq/BP/m5.go.bp.v2026.1.Mm.symbols.gmt"
OBO_PATH = "go.obo"


OUT_MAPPING = "msigdb_to_go.tsv"
OUT_UNMATCHED = "msigdb_unmatched.tsv"

BASE_URL = (
    "https://www.gsea-msigdb.org/gsea/msigdb/"
    "mouse/geneset/{}.json"
)


def get_valid_bp_ids(obo_path):
    """获取 go.obo 中所有有效的 BP GO ID。"""
    valid_ids = set()
    current = None

    def save_term():
        if (
            current
            and current.get("id")
            and current.get("namespace") == "biological_process"
            and not current.get("obsolete", False)
        ):
            valid_ids.add(current["id"])

    with open(obo_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line == "[Term]":
                save_term()
                current = {
                    "id": None,
                    "namespace": None,
                    "obsolete": False,
                }
            elif line.startswith("["):
                save_term()
                current = None
            elif current is not None:
                if line.startswith("id: "):
                    current["id"] = line[4:].strip()
                elif line.startswith("namespace: "):
                    current["namespace"] = line[11:].strip()
                elif line == "is_obsolete: true":
                    current["obsolete"] = True

    save_term()
    return valid_ids


def read_gmt(gmt_path):
    """读取 GMT 中的基因集名称和 URL。"""
    records = []

    with open(gmt_path, encoding="utf-8") as f:
        for line in f:
            fields = line.rstrip("\r\n").split("\t")

            if len(fields) >= 3:
                records.append({
                    "gene_set": fields[0],
                    "url": fields[1],
                })

    return records


def main():
    valid_bp_ids = get_valid_bp_ids(OBO_PATH)
    records = read_gmt(GMT_PATH)

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})

    matched = []
    unmatched = []

    for i, record in enumerate(records, 1):
        name = record["gene_set"]

        # 使用 GMT 中的 URL，避免手动拼接时出现路径差异
        url = BASE_URL.format(name)

        try:
            response = session.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()

            # 关键修正：JSON 顶层以基因集名称为 key
            gene_set_data = data.get(name)

            if gene_set_data is None:
                unmatched.append({
                    "gene_set": name,
                    "reason": "Gene set key not found in JSON",
                    "go_ids": "",
                })
                continue

            exact_source = gene_set_data.get("exactSource", "")
            go_ids = sorted(set(re.findall(
                r"GO:\d{7}", str(exact_source)
            )))

            # GOBP 文件只接受 BP term
            valid_ids = [go_id for go_id in go_ids
                         if go_id in valid_bp_ids]

            if valid_ids:
                for go_id in valid_ids:
                    matched.append({
                        "gene_set": name,
                        "go_id": go_id,
                        "status": "valid_BP",
                    })
            else:
                unmatched.append({
                    "gene_set": name,
                    "reason": (
                        "No exactSource GO ID"
                        if not go_ids
                        else "GO ID not found as valid BP term in OBO"
                    ),
                    "go_ids": ";".join(go_ids),
                })

        except Exception as e:
            unmatched.append({
                "gene_set": name,
                "reason": str(e),
                "go_ids": "",
            })

        if i % 50 == 0:
            print(f"Processed {i}/{len(records)}")

        time.sleep(0.1)

    with open(OUT_MAPPING, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["gene_set", "go_id", "status"],
            delimiter="\t",
        )
        writer.writeheader()
        writer.writerows(matched)

    with open(OUT_UNMATCHED, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["gene_set", "reason", "go_ids"],
            delimiter="\t",
        )
        writer.writeheader()
        writer.writerows(unmatched)

    print("\nFinished")
    print("GMT gene sets:", len(records))
    print("Valid BP mappings:", len(matched))
    print("Unmatched:", len(unmatched))
    print("Mapping file:", OUT_MAPPING)
    print("Unmatched file:", OUT_UNMATCHED)



if __name__ == "__main__":
    main()


Processed 50/7781
Processed 100/7781
Processed 150/7781
Processed 200/7781
Processed 250/7781
Processed 300/7781
Processed 350/7781
Processed 400/7781
Processed 450/7781
Processed 500/7781
Processed 550/7781
Processed 600/7781
Processed 650/7781
Processed 700/7781
Processed 750/7781
Processed 800/7781
Processed 850/7781
Processed 900/7781
Processed 950/7781
Processed 1000/7781
Processed 1050/7781
Processed 1100/7781
Processed 1150/7781
Processed 1200/7781
Processed 1250/7781
Processed 1300/7781
Processed 1350/7781
Processed 1400/7781
Processed 1450/7781
Processed 1500/7781
Processed 1550/7781
Processed 1600/7781
Processed 1650/7781
Processed 1700/7781
Processed 1750/7781
Processed 1800/7781
Processed 1850/7781
Processed 1900/7781
Processed 1950/7781
Processed 2000/7781
Processed 2050/7781
Processed 2100/7781
Processed 2150/7781
Processed 2200/7781
Processed 2250/7781
Processed 2300/7781
Processed 2350/7781
Processed 2400/7781
Processed 2450/7781
Processed 2500/7781
Processed 2550/7781


In [8]:
#making GO BP Gene annotation index file
import pandas as pd

# =========================
# 1. 文件路径
# =========================
BP_PATH = "mESC_BP_index.csv"
MAPPING_PATH = "msigdb_to_go.tsv"

OUT_MAPPED = "mESC_BP_GO_mapped.csv"
OUT_DETAILS = "mESC_BP_mapping_details.tsv"
OUT_UNMAPPED = "mESC_BP_unmapped.csv"


# =========================
# 2. 读取文件
# =========================
# mESC_BP.csv: 行为 GOBP 基因集，列为基因符号
bp = pd.read_csv(BP_PATH, index_col=0)

# 只包含前50个基因集的映射文件
mapping = pd.read_csv(MAPPING_PATH, sep="\t")

print("mESC_BP 原始矩阵形状:", bp.shape)
print("映射文件记录数:", len(mapping))


# =========================
# 3. 检查并清理映射表
# =========================
required_cols = {"gene_set", "go_id"}
if not required_cols.issubset(mapping.columns):
    raise ValueError(
        f"映射文件必须包含列 {required_cols}，"
        f"实际列为 {mapping.columns.tolist()}"
    )

mapping = mapping[["gene_set", "go_id"]].dropna()
mapping["gene_set"] = mapping["gene_set"].astype(str).str.strip()
mapping["go_id"] = mapping["go_id"].astype(str).str.strip()

# 同一基因集若对应多个 GO ID，保留每一条对应关系
mapping = mapping.drop_duplicates()


# =========================
# 4. 将 mESC_BP 行名映射为 GO ID
# =========================
# 原始基因集名称 -> GO ID
gene_set_to_go = (
    mapping.groupby("gene_set")["go_id"]
    .apply(list)
    .to_dict()
)

mapped_rows = []
unmapped_names = []

for gene_set, row in bp.iterrows():
    go_ids = gene_set_to_go.get(str(gene_set).strip())

    if not go_ids:
        unmapped_names.append(gene_set)
        continue

    for go_id in go_ids:
        mapped_rows.append({
            "gene_set": gene_set,
            "go_id": go_id,
            "values": row.to_numpy()
        })


# =========================
# 5. 保存映射明细
# =========================
details = pd.DataFrame([
    {
        "gene_set": item["gene_set"],
        "go_id": item["go_id"]
    }
    for item in mapped_rows
])

details.to_csv(
    OUT_DETAILS,
    sep="\t",
    index=False
)


# =========================
# 6. 生成 GO ID × gene 矩阵
# =========================
if mapped_rows:
    mapped_df = pd.DataFrame(
        [item["values"] for item in mapped_rows],
        columns=bp.columns
    )

    mapped_df.index = [
        item["go_id"] for item in mapped_rows
    ]
    mapped_df.index.name = "GO_ID"

    # 多个基因集可能映射到同一个 GO ID。
    # 对二值注释矩阵按列取 max，保留任一来源中的阳性注释。
    mapped_df = mapped_df.groupby(level=0, sort=False).max()

    mapped_df.to_csv(OUT_MAPPED)

else:
    print("警告：没有任何行成功映射，请检查映射文件。")


# =========================
# 7. 保存未映射行
# =========================
bp.loc[unmapped_names].to_csv(OUT_UNMAPPED)


# =========================
# 8. 输出统计
# =========================
print("\n映射完成")
print("原始基因集数:", len(bp))
print("成功映射的原始基因集数:",
      bp.index.nunique() - len(set(unmapped_names)))
print("未映射的原始基因集数:", len(set(unmapped_names)))

if mapped_rows:
    print("映射后的唯一 GO ID 数:", mapped_df.shape[0])
    print("映射矩阵形状:", mapped_df.shape)

print("输出文件:")
print("  ", OUT_MAPPED)
print("  ", OUT_DETAILS)
print("  ", OUT_UNMAPPED)


mESC_BP 原始矩阵形状: (7782, 1120)
映射文件记录数: 7453

映射完成
原始基因集数: 7782
成功映射的原始基因集数: 7453
未映射的原始基因集数: 329
映射后的唯一 GO ID 数: 7453
映射矩阵形状: (7453, 1120)
输出文件:
   mESC_BP_GO_mapped.csv
   mESC_BP_mapping_details.tsv
   mESC_BP_unmapped.csv
